In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
#lets first import all the data we are gonna write a small script to do so 

from pathlib import Path

#below is the path where all 9 datasets are stored there in this folder

#the r tells the python dont treat backlashes as escape characters
raw_data_dir = Path(r"D:\Projects\Retails-Analytics-Data-Warehouse\data\raw")

#converting all our csvs to list so that we run loop over it
#glob is used to tell the script go into this folder and find everything that 
#has .csv
csv_files = list(raw_data_dir.glob("*.csv"))

print(f"Found {len(csv_files)} csv files.\n")

for file in csv_files:

    df = pd.read_csv(file)

    print("="*50)
    print(f"File: {file.name}")
    print("="*50)

    print(f"Rows: {df.shape[0]:,}")
    print(f"columns: {df.shape[1]}")

    print("\ncolumns")

    print(df.columns.tolist())

    print("\n Data Types")
    print(df.dtypes)

    print("\nMissing Values: ")

    print(df.isnull().sum())

    print("\nDuplicate Rows: ")
    print(df.duplicated().sum())

    print()

Found 9 csv files.

File: olist_customers_dataset.csv
Rows: 99,441
columns: 5

columns
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

 Data Types
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Missing Values: 
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate Rows: 
0

File: olist_geolocation_dataset.csv
Rows: 1,000,163
columns: 5

columns
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

 Data Types
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Mi

In [11]:
#Primary key validation

print("\n" + "=" * 60)
print("PRIMARY KEY VALIDATION")
print("=" * 60)

primary_keys = {
    "olist_customers_dataset.csv": "customer_id",
    "olist_orders_dataset.csv": "order_id",
    "olist_products_dataset.csv": "product_id",
    "olist_sellers_dataset.csv": "seller_id",
    "olist_order_reviews_dataset.csv": "review_id"
}

for file_name, primary_key in primary_keys.items():

    file_path = raw_data_dir / file_name
    df = pd.read_csv(file_path)

    total_rows = len(df)
    unique_values = df[primary_key].nunique()
    null_values = df[primary_key].isnull().sum()

    print(f"\n{file_name}")
    print(f"Primary Key: {primary_key}")
    print(f"Total rows: {total_rows:,}")
    print(f"Unique values: {unique_values:,}")
    print(f"Null values: {null_values}")

    if total_rows == unique_values and null_values == 0:
        print("STATUS: PASS")
    else:
        print("STATUS: FAIL")


PRIMARY KEY VALIDATION

olist_customers_dataset.csv
Primary Key: customer_id
Total rows: 99,441
Unique values: 99,441
Null values: 0
STATUS: PASS

olist_orders_dataset.csv
Primary Key: order_id
Total rows: 99,441
Unique values: 99,441
Null values: 0
STATUS: PASS

olist_products_dataset.csv
Primary Key: product_id
Total rows: 32,951
Unique values: 32,951
Null values: 0
STATUS: PASS

olist_sellers_dataset.csv
Primary Key: seller_id
Total rows: 3,095
Unique values: 3,095
Null values: 0
STATUS: PASS

olist_order_reviews_dataset.csv
Primary Key: review_id
Total rows: 99,224
Unique values: 98,410
Null values: 0
STATUS: FAIL


In [12]:
#so the reviews dataset failed here on primary key now we also desgined it as primary key
#in the database and this should be unique lets investigate it why its not unique

print("\n" + "=" * 60)
print("REVIEW ID DUPLICATE INVESTIGATION")
print("=" * 60)


#here we have imported the raw_data_dir above which contains all the files so 
# here we say take out only the order_reviews dataset 

reviews = pd.read_csv(raw_data_dir / "olist_order_reviews_dataset.csv")


#and then here we are taking the only column which we take it as a primary key
duplicate_review_ids = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

print(f"Rows involved in duplicated review_ids: {len(duplicate_review_ids):,}")
print(f"Number of duplicated review_ids: {duplicate_review_ids['review_id'].nunique():,}")

print("\nSample duplicated review_ids:")
print(duplicate_review_ids.head(20))


REVIEW ID DUPLICATE INVESTIGATION
Rows involved in duplicated review_ids: 1,603
Number of duplicated review_ids: 789

Sample duplicated review_ids:
                              review_id                          order_id  \
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195   
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa   
90677  0115633a9c298b6a98bcbe4eee75345f  78a4201f58af3463bdab842eea4bc801   
63193  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
92876  0174caf0ee5964646040cd94e15ac95e  f93a732712407c02dce5dd5088d0f47b   
57280  0174caf0ee5964646040cd94e15ac95e  74db91e33b4e1fd865356c89a61abf1f   
54832  017808d29fd1f942d97e50184dfb4c13  8daaa9e99d60fbba579cc1c3e3bfae01   
99167  017808d29fd1f942d97e50184dfb4c13  b1461c8882153b5fe68307c46a506e39   
20621  0254bd905dc677a6078990aad3331a36  5bf226cf882c5bf4247f89a97c86f273   
96080  0254bd905dc677a6078990aad3331a36  331b367bdd766f3d1cf518777317b5d9   
8971

In [18]:
print("\n" + "=" * 60)
print("REVIEW KEY INVESTIGATION")
print("=" * 60)

# Check whether review_id + order_id is unique
review_order_duplicates = reviews.duplicated(
    subset=["review_id", "order_id"],
    keep=False
)

print(
    "Rows with duplicated (review_id, order_id):",
    review_order_duplicates.sum()
)

duplicated_combinations = reviews.loc[
    review_order_duplicates,
    ["review_id", "order_id"]
].drop_duplicates()

print(
    "Unique duplicated (review_id, order_id) combinations:",
    len(duplicated_combinations)
)


REVIEW KEY INVESTIGATION
Rows with duplicated (review_id, order_id): 0
Unique duplicated (review_id, order_id) combinations: 0


In [ ]:
#NO WHERE IS something important even though review_id alone is duplicated, the combination of review_id +order_id 
#is unique for every row

#so what we got is review_id alone is not unique but review_id+order_id is unique

#now we declared this review_id as primary key but it is not unique so what are we gonna do is we
#keep this as composite key so it will be review_id +order_id which will act like composite 
#key 

#this is not becuase out review_id is not unique then lets go for composite key so it just not random
#rather we investigated this order_id represent the order associated with the review


In [22]:
print("=" * 50)
print("COMPOSITE KEY VALIDATION")

print("=" *50)

order_items = pd.read_csv(
    raw_data_dir / "olist_order_items_dataset.csv"
)


#subset is only order_id and order_item_id as we are not gonna check all the 
#duplicates just these two as we are investigating these two as we taken these two 
#as our composite_id
order_items_duplicates = order_items.duplicated(
    subset=["order_id", "order_item_id"],
    keep=False
)

print("Order Items")

print("\nOrder Items")
print("Key: (order_id, order_item_id)")
print(
    "Duplicated key rows:",
    order_items_duplicates.sum()
)


#next is for payments table

payments = pd.read_csv(
    raw_data_dir / "olist_order_payments_dataset.csv"
)

payment_duplicates = payments.duplicated(
    subset=["order_id", "payment_sequential"],
    keep=False
)

print("\nPayments")
print("Key: (order_id, payment_sequential)")
print(
    "Duplicated key rows:",
    payment_duplicates.sum()
)



COMPOSITE KEY VALIDATION
Order Items

Order Items
Key: (order_id, order_item_id)
Duplicated key rows: 0

Payments
Key: (order_id, payment_sequential)
Duplicated key rows: 0


In [25]:
print("=" *60)
print("Foreign key validation")
print("="*60)

customers = pd.read_csv(raw_data_dir / "olist_customers_dataset.csv")
orders = pd.read_csv(  raw_data_dir / "olist_orders_dataset.csv"
)

#now checking whether every order.customer_id exists in customers.cusotmer_id

#~ we used that symbol for "not" in python

invalid_customer_ids = orders.loc[~orders["customer_id"].isin(customers["customer_id"]),
    "customer_id"]


print("orders --> customers")
print("Foreign Key: orders.customer_id")
print("Referenced Key: customers.customer_id")
print("Invalid foreign keys:", invalid_customer_ids.nunique())


if invalid_customer_ids.nunique() == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

Foreign key validation
orders --> customers
Foreign Key: orders.customer_id
Referenced Key: customers.customer_id
Invalid foreign keys: 0
STATUS: PASS


In [26]:
#Another foreign key relationship between the two tables

order_items = pd.read_csv(
    raw_data_dir / "olist_order_items_dataset.csv"
)

invalid_order_ids = order_items.loc[
    ~order_items["order_id"].isin(orders["order_id"]),
    "order_id"
]

print("\nOrder Items --> Orders")
print("Foreign Key: order_items.order_id")
print("Referenced Key: orders.order_id")
print("Invalid foreign keys:", invalid_order_ids.nunique())

if invalid_order_ids.nunique() == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")


Order Items --> Orders
Foreign Key: order_items.order_id
Referenced Key: orders.order_id
Invalid foreign keys: 0
STATUS: PASS


In [27]:
#now next is order_item and products which are tied through product_id also


products = pd.read_csv(
    raw_data_dir / "olist_products_dataset.csv"
)


#this line below is the heart of the code 
invalid_product_ids = order_items.loc[
    ~order_items["product_id"].isin(products["product_id"]),
    "product_id"
]

print("\nOrder Items --> Products")
print("Foreign Key: order_items.product_id")
print("Referenced Key: products.product_id")
print("Invalid foreign keys:", invalid_product_ids.nunique())

if invalid_product_ids.nunique() == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")


Order Items --> Products
Foreign Key: order_items.product_id
Referenced Key: products.product_id
Invalid foreign keys: 0
STATUS: PASS


In [28]:
#nexxt tables are order_itmes and sellers they are tied by seller_id does every seller 
#reference in order_items actually exist in the seller tables

In [29]:


sellers = pd.read_csv(
    raw_data_dir / "olist_sellers_dataset.csv"
)


#the below line tells find seller IDS in order_items that do not exist in sellers
invalid_seller_ids = order_items.loc[
    ~order_items["seller_id"].isin(sellers["seller_id"]),
    "seller_id"
]

print("\nOrder Items --> Sellers")
print("Foreign Key: order_items.seller_id")
print("Referenced Key: sellers.seller_id")
print("Invalid foreign keys:", invalid_seller_ids.nunique())

if invalid_seller_ids.nunique() == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")


Order Items --> Sellers
Foreign Key: order_items.seller_id
Referenced Key: sellers.seller_id
Invalid foreign keys: 0
STATUS: PASS


In [ ]:
#next tables are orders payments the foreign key is order_id

#payment.order_id reference orders.order_id 

In [31]:


payments = pd.read_csv(
    raw_data_dir / "olist_order_payments_dataset.csv"
)

invalid_payment_order_ids = payments.loc[
    ~payments["order_id"].isin(orders["order_id"]),
    "order_id"
]

print("\nPayments --> Orders")
print("Foreign Key: payments.order_id")
print("Referenced Key: orders.order_id")
print(
    "Invalid foreign keys:",
    invalid_payment_order_ids.nunique()
)

if invalid_payment_order_ids.nunique() == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")


Payments --> Orders
Foreign Key: payments.order_id
Referenced Key: orders.order_id
Invalid foreign keys: 0
STATUS: PASS


In [32]:

reviews = pd.read_csv(
    raw_data_dir / "olist_order_reviews_dataset.csv"
)

invalid_review_order_ids = reviews.loc[
    ~reviews["order_id"].isin(orders["order_id"]),
    "order_id"
]

print("\nReviews --> Orders")
print("Foreign Key: reviews.order_id")
print("Referenced Key: orders.order_id")
print(
    "Invalid foreign keys:",
    invalid_review_order_ids.nunique()
)

if invalid_review_order_ids.nunique() == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")


Reviews --> Orders
Foreign Key: reviews.order_id
Referenced Key: orders.order_id
Invalid foreign keys: 0
STATUS: PASS


In [ ]:
#so the above relationship is valid because it passes the foregn key condition

In [4]:


geolocation = pd.read_csv(
    raw_data_dir / "olist_geolocation_dataset.csv"
)

print("\n" + "=" * 60)
print("GEOLOCATION KEY INVESTIGATION")
print("=" * 60)

print("Total rows:", len(geolocation))
print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)
print(
    "Duplicated ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].duplicated().sum()
)


GEOLOCATION KEY INVESTIGATION
Total rows: 1000163
Unique ZIP prefixes: 19015
Duplicated ZIP prefixes: 981148


In [5]:


duplicated_zip_prefixes = geolocation[
    geolocation["geolocation_zip_code_prefix"].duplicated(
        keep=False
    )
]

print("\nExamples of repeated ZIP prefixes:")
print(
    duplicated_zip_prefixes
    .sort_values("geolocation_zip_code_prefix")
    .head(20)
)


Examples of repeated ZIP prefixes:
      geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
429                          1001       -23.550498       -46.634338   
519                          1001       -23.551337       -46.634027   
575                          1001       -23.549779       -46.633957   
596                          1001       -23.550498       -46.634338   
206                          1001       -23.550498       -46.634338   
583                          1001       -23.551337       -46.634027   
608                          1001       -23.550263       -46.634196   
639                          1001       -23.550498       -46.634338   
1384                         1001       -23.549292       -46.633559   
1435                         1001       -23.549292       -46.633559   
818                          1001       -23.551337       -46.634027   
235                          1001       -23.550642       -46.634410   
326                          1001       -

In [6]:
#here we are grouping these zip_code_prefix to find how many unique coordinated are there in each zipp_code_prfix


print("\nUnique locations per ZIP prefix:")

zip_location_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        unique_lat=("geolocation_lat", "nunique"),
        unique_lng=("geolocation_lng", "nunique"),
        unique_cities=("geolocation_city", "nunique"),
        unique_states=("geolocation_state", "nunique")
    )
)

print(zip_location_counts.head(20))


Unique locations per ZIP prefix:
                             unique_lat  unique_lng  unique_cities  \
geolocation_zip_code_prefix                                          
1001                                 10          10              2   
1002                                  6           6              2   
1003                                 10          10              2   
1004                                 14          14              2   
1005                                 11          11              2   
1006                                  7           7              2   
1007                                 15          15              2   
1008                                 12          12              2   
1009                                  7           7              2   
1010                                  5           5              2   
1011                                  3           3              2   
1012                                 13          13     

In [9]:
#going forward with another lookup table translation table which translates the category name from portuguese to engish
#so it is 

products = pd.read_csv(
    raw_data_dir / "olist_products_dataset.csv"
)

products.head(5)


#look the names are in portuguese  o for english naem we look into translation table

category_translation = pd.read_csv(
    raw_data_dir / "product_category_name_translation.csv"
)

category_translation.head(5)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [10]:
#investigation 

print("\n" + "=" * 60)
print("CATEGORY TRANSLATION KEY INVESTIGATION")
print("=" * 60)

print(
    "Translation rows:",
    len(category_translation)
)

print(
    "Unique category names:",
    category_translation["product_category_name"].nunique()
)

print(
    "Duplicate category names:",
    category_translation[
        "product_category_name"
    ].duplicated().sum()
)


CATEGORY TRANSLATION KEY INVESTIGATION
Translation rows: 71
Unique category names: 71
Duplicate category names: 0


In [ ]:
#so for this we have 71 unque rows representing names of te product and thye are also unique so this is a good matching column
#now next thing we are gonna check out whether  every category that appears in the product tables acutally exist in the translation table

In [11]:

product_categories = products["product_category_name"].dropna()

missing_translations = product_categories[
    ~product_categories.isin(
        category_translation["product_category_name"]
    )
]

print("\n" + "=" * 60)
print("CATEGORY TRANSLATION COVERAGE")
print("=" * 60)

print(
    "Product categories without translation:",
    missing_translations.nunique()
)

print("\nMissing categories:")

print(
    missing_translations.unique()
)


CATEGORY TRANSLATION COVERAGE
Product categories without translation: 2

Missing categories:
['pc_gamer' 'portateis_cozinha_e_preparadores_de_alimentos']


## Data Tranformation

In [ ]:
#now here we are getting our datasets ready to be imported to the database  